# BTYD Modelling — Customer Purchase Prediction

**Workflow:**
1. Load & prepare 2021 training data
2. Compute BTYD inputs (frequency, recency, T, monetary)
3. Fit BG/NBD model → predict purchases & probability alive
4. Fit Gamma-Gamma model → predict expected order value
5. Combine → predicted 90-day spend per customer
6. Validate against actual 2022 Q1 data

---
## 1. Imports & Data Loading

In [16]:
import pandas as pd
from lifetimes import BetaGeoFitter, GammaGammaFitter

In [17]:
# Training data: full 2021 customer history
df_2021 = pd.read_parquet("../data/processed/cleaned_2021_customer_data.parquet")

# Holdout data: 2022 actuals for model validation
df_2022 = pd.read_parquet("../data/processed/cleaned_2022_customer_data.parquet")

---
## 2. Prepare BTYD Inputs

BTYD models require four inputs per customer:
- **frequency** — number of *repeat* purchases (total - 1)
- **recency** — days between first and last purchase
- **T (tenure)** — days between first purchase and end of observation period
- **monetary** — average order value (repeat buyers only, used in Gamma-Gamma)

In [18]:
# Frequency: repeat purchases only (BTYD convention excludes the first transaction)
df_2021['frequency'] = df_2021['transaction_count'] - 1

# Recency: days between first and last purchase
df_2021['recency'] = (
    pd.to_datetime(df_2021['last_purchase_date']) - pd.to_datetime(df_2021['first_purchase_date'])
).dt.days


# T (Tenure): days from first purchase to end of observation window
observation_end = pd.to_datetime(df_2021['last_purchase_date']).max()
df_2021['T'] = (observation_end - pd.to_datetime(df_2021['first_purchase_date'])).dt.days


# Monetary: average order value (used later for Gamma-Gamma, repeat buyers only)
df_2021['monetary'] = df_2021['avg_order_value']

df_2021[['customer_id', 'frequency', 'recency', 'T', 'monetary']].head()

,customer_id,frequency,recency,T,monetary
0,21644,0,0,364,58.543
1,73192,0,0,364,74.030
2,9779,0,0,364,4.094
3,98867,0,0,363,14.923
4,48935,0,0,363,35.029


---
## 3. BG/NBD Model — Purchase Frequency & Probability Alive

The BetaGeo (BG/NBD) model uses frequency, recency, and tenure to:
- Estimate the probability a customer is still active
- Predict expected number of future purchases

In [19]:
bgf = BetaGeoFitter(penalizer_coef=0.0)
bgf.fit(df_2021['frequency'], df_2021['recency'], df_2021['T'])

c:\Users\NewAdminUser\OneDrive\Documents\DANNY DATA\Ecommerce CLV Prediction\.venv\lib\site-packages\pandas\core\arraylike.py:396: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


<lifetimes.BetaGeoFitter: fitted with 36086 subjects, a: 0.00, alpha: 32.34, b: 8810.85, r: 0.66>

In [20]:
# Probability each customer is still active
df_2021['prob_alive'] = bgf.conditional_probability_alive(
    df_2021['frequency'], df_2021['recency'], df_2021['T']
)

# Expected purchases in the next 90 days
PREDICTION_HORIZON = 90
df_2021['predicted_purchases_90d'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    PREDICTION_HORIZON, df_2021['frequency'], df_2021['recency'], df_2021['T']
)

df_2021[['customer_id', 'prob_alive', 'predicted_purchases_90d']].sample(5)

,customer_id,prob_alive,predicted_purchases_90d
679,11437,1.0,0.166892
4780,88205,1.0,0.381639
6941,66704,1.0,0.354510
18949,10532,1.0,1.917308
26115,80999,1.0,3.075451


---
## 4. Gamma-Gamma Model — Expected Order Value

The Gamma-Gamma model estimates how much a customer will spend per transaction.

> **Important:** Gamma-Gamma only applies to repeat buyers (frequency > 0).
> One-time buyers are excluded here — the model has no repeat spend signal for them.

In [21]:
# Gamma-Gamma requires repeat purchasers only
df_repeat = df_2021[df_2021['frequency'] > 0].copy()

print(f"Total customers:         {len(df_2021):,}")
print(f"Repeat buyers (Gamma input): {len(df_repeat):,}")
print(f"One-time buyers excluded: {len(df_2021) - len(df_repeat):,}")

Total customers:         36,086
Repeat buyers (Gamma input): 26,707
One-time buyers excluded: 9,379


In [22]:
# Gamma-Gamma assumption check: frequency and monetary should be weakly correlated
# Correlation should be close to 0 — if high, the model assumptions are violated
print(df_repeat[['frequency', 'monetary']].corr())

           frequency  monetary
frequency   1.000000 -0.007102
monetary   -0.007102  1.000000


In [23]:
ggf = GammaGammaFitter(penalizer_coef=0)
ggf.fit(df_repeat['frequency'], df_repeat['monetary'])

<lifetimes.GammaGammaFitter: fitted with 26707 subjects, p: 5.09, q: 1.42, v: 9.69>

In [24]:
# Expected average order value per repeat customer
df_repeat['expected_avg_order_value'] = ggf.conditional_expected_average_profit(
    df_repeat['frequency'],
    df_repeat['monetary']
)

---
## 5. Combine — Predicted 90-Day Spend

Merge predicted purchase count (from BG/NBD) with expected order value (from Gamma-Gamma).

`predicted_total_spend_90d = predicted_purchases_90d × expected_avg_order_value`

In [25]:
# Merge BG/NBD purchase predictions into the repeat buyers dataframe
df_repeat = df_repeat.merge(
    df_2021[['customer_id', 'predicted_purchases_90d', 'prob_alive']],
    on='customer_id',
    how='left',
    suffixes=('', '_bgf')  # avoid column collision if prob_alive already exists
)

# Predicted total spend = predicted transaction count × expected order value
df_repeat['predicted_total_spend_90d'] = (
    df_repeat['predicted_purchases_90d'] * df_repeat['expected_avg_order_value']
)

df_repeat[[
    'customer_id',
    'prob_alive',
    'predicted_purchases_90d',
    'expected_avg_order_value',
    'predicted_total_spend_90d'
]].head()

,customer_id,prob_alive,predicted_purchases_90d,expected_avg_order_value,predicted_total_spend_90d
0,82104,1.0,0.384595,35.099410,13.499061
1,57056,1.0,0.402249,141.287851,56.832945
2,73701,1.0,0.406641,25.285946,10.282312
3,96781,1.0,0.385591,18.965942,7.313088
4,78400,1.0,0.425213,1502.664886,638.951979


---
## 6. Validation — Predicted vs Actual (2022 Q1)

Compare model predictions against actual customer behaviour in the first 90 days of 2022.

In [26]:
# Filter 2022 data to first 90 days (Jan 1 – Mar 31)
cutoff_date = pd.to_datetime('2021-12-31') + pd.Timedelta(days=90)

df_2022['last_purchase_date'] = pd.to_datetime(df_2022['last_purchase_date'])

df_2022_90d = df_2022[df_2022['last_purchase_date'] <= cutoff_date]

In [27]:
# Build evaluation table: predictions vs actuals side-by-side
eval_df = df_repeat[[
    'customer_id',
    'predicted_purchases_90d',
    'expected_avg_order_value',
    'predicted_total_spend_90d'
]].merge(
    df_2022_90d[['customer_id', 'transaction_count', 'avg_order_value', 'total_spent']],
    on='customer_id',
    how='left'
).rename(columns={
    'transaction_count': 'actual_purchases_90d',
    'avg_order_value':   'actual_avg_order_value',
    'total_spent':       'actual_total_spend_90d'
}).fillna(0)

eval_df.head()

,customer_id,predicted_purchases_90d,expected_avg_order_value,predicted_total_spend_90d,actual_purchases_90d,actual_avg_order_value,actual_total_spend_90d
0,82104,0.384595,35.099410,13.499061,0.0,0.000,0.000
1,57056,0.402249,141.287851,56.832945,0.0,0.000,0.000
2,73701,0.406641,25.285946,10.282312,1.0,28.073,28.073
3,96781,0.385591,18.965942,7.313088,0.0,0.000,0.000
4,78400,0.425213,1502.664886,638.951979,0.0,0.000,0.000
